In [ ]:
#####| Fixed scenario condition    | What changes across seeds                                                                                |
#####| --------------------------- | -------------------------------------------------------------------------------------------------------- |
#####| `max_inter=0` and `L_var=0` | **No differences** (`inter=0`; `sigmas`/`lambdas` unchanged)                                             |
#####| `max_inter>0` and `L_var=0` | **Only** the interaction matrix `inter` differs                                                          |
#####| `max_inter=0` and `L_var>0` | **Only** `sigmas` / `lambdas` differ (and they affect **t ≥ 4h** since phase 1 uses `sigma=0, lambda=0`) |
#####| `max_inter>0` and `L_var>0` | Both `inter` and `sigmas` / `lambdas` differ                                                             |


# Mechanistic model for data generation under different scenarios
import numpy as np   #for numerical computing. It is used to handle arrays and matrix operations
from scipy.integrate import odeint  #for solving ordinary differential equations (ODEs)


# =============== define functions ===============
def generate_valid_random(N): ##Generate a random array of size N from a normal distribution such that its mean is less than or equal to 0.1 and its standard deviation is less than or equal to 1.2. If these conditions are not met, the array is regenerated
    while True:
        r = np.random.randn(N)  #randn is a function in NumPy that generates random numbers from a standard normal distribution (i.e., normal distribution with mean 0 and variance 1)
        if abs(np.mean(r)) <= 0.1 and np.std(r) <= 1.2:
            return r
            
def param_generator_lv(N, connectedness, neg_frac, max_neg, max_pos, rng): 
    # Generate an interaction matrix using RNG
    indsPerm = rng.permutation(N * (N - 1))
    inter_init = np.zeros((N, N - 1))
    
    neg_count = int(np.ceil(N * (N - 1) * connectedness * neg_frac))
    pos_count = int(np.floor(N * (N - 1) * connectedness * (1 - neg_frac)))
    
    inter_init.flat[indsPerm[:neg_count]] = -rng.random(neg_count) * max_neg
    inter_init.flat[indsPerm[neg_count:neg_count+pos_count]] = rng.random(pos_count) * max_pos
    
    inter = np.zeros((N, N)) 
    for i in range(N):
        mask = np.ones(N, dtype=bool)
        mask[i] = False
        inter[i, mask] = inter_init[i, :]
    
    return [inter, inter * (inter < 0), inter * (inter > 0)]


def calculate_growth_rates(x, params, max_Gs, sigma, lambda_val):
    # return G_i-L_i
    return (max_Gs * (1 - x) + params[0] @ x) * (1 - sigma) - lambda_val

def calculate_effective_growth_rates(x, params, max_Gs):
    # return G_i
    return (max_Gs * (1 - x) + params[0] @ x) 

def calculate_lysis_rates(x, params, max_Gs, sigma, lambda_val):
    # return L_i
    return (max_Gs * (1 - x) + params[0] @ x) * sigma + lambda_val


def core_ode_lv_residue(state, t, inter, max_Gs, sigma, lambda_val, li, de):

    S = state[:N]
    R = state[N:]

    # effective growth rate
    G = max_Gs * (1 - S) + inter @ S

    # phase switch
    if t < 4:
        sigma_eff = np.zeros_like(sigma)
        lambda_eff = np.zeros_like(lambda_val)
    else:
        sigma_eff = sigma
        lambda_eff = lambda_val

    # lysis rate
    L = sigma_eff * G + lambda_eff

    # dS/dt
    dSdt = S * (G - L)

    # dR/dt
    dRdt = li * L * S - de * R

    # enforce R >= 0
    mask = (R <= 0) & (dRdt < 0)
    dRdt[mask] = 0

    return np.concatenate([dSdt, dRdt])


def simulate_timecourse_with_rates(x0, tend, params, max_Gs, sigma, lambda_val, li, de):

    t = np.linspace(0, tend, 111)

    # initial state
    S0 = x0
    R0 = np.zeros(N)

    state0 = np.concatenate([S0, R0])

    solution = odeint(
        core_ode_lv_residue,
        state0,
        t,
        args=(params[0], max_Gs, sigma, lambda_val, li, de),
        atol=1e-9
    )

    # split solution
    y = solution[:, :N]
    R = solution[:, N:]

    growth_rates = np.zeros_like(y)
    effective_growth_rates = np.zeros_like(y)
    lysis_rates = np.zeros_like(y)
    Bi_residue = np.zeros_like(y)
    net_rate_resi = np.zeros_like(y)
    lysis_rate_with_resi = np.zeros_like(y)


    for i in range(len(t)):

        effective_growth_rates[i] = calculate_effective_growth_rates(y[i], params, max_Gs)
        
        if t[i] < 4:
            lysis_rates[i] = np.zeros_like(y[i])
        else:
            lysis_rates[i] = calculate_lysis_rates(y[i], params, max_Gs, sigma, lambda_val)

        growth_rates[i] = effective_growth_rates[i] - lysis_rates[i]
        # ===== residue quantities =====
        Bi_residue[i] = y[i] + R[i]
        numerator = y[i] * (effective_growth_rates[i] - lysis_rates[i] + li * lysis_rates[i]) - de * R[i]
        denom = np.maximum(Bi_residue[i], 1e-12)
        net_rate_resi[i] = numerator / denom
        lysis_rate_with_resi[i] = effective_growth_rates[i] - net_rate_resi[i]

    return t, y, growth_rates, effective_growth_rates, lysis_rates,R, Bi_residue, net_rate_resi, lysis_rate_with_resi


# ====================== Parameter settings ======================
N = 57
x0 = np.ones(N) / (100 * N)

tend = 11 #Simulation time (11 hours).
max_Gs = np.linspace(0.7, 1.5, N) #Maximum growth rates for each species, linearly spaced between 0.7 and 1.5.
sigma = 1.5 * np.ones(N)        # a
lambda_val = 0.1 * np.ones(N)   # b

connectedness = 1 #density of interactions between species (1 means fully connected).
neg_frac = 1
max_neg = 0
max_pos = max_neg * 0.5



# ====================== Initialize output arrays ====================== 
n_time = tend*10+1 # 10 time points per hour
n_species = 57
# parameter set
scenario_params = [{'L_var': lv, 'max_inter': mi} 
                  for lv in [0, 0.1, 0.2] 
                  for mi in [1, 5, 10]]
n_scenarios = len(scenario_params) 

n_seeds = 100  # keep this to define the array size

# Initialize output arrays once for all seeds
outputs = {
    'population_data': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'growth_rates': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'effective_growth_rates': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'lysis_rates': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'relative_abundance': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'residue': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'biomass_with_resid': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'net_rate_with_resid': np.zeros((n_time, n_species, n_seeds, n_scenarios)),
    'lysis_rate_with_resid': np.zeros((n_time, n_species, n_seeds, n_scenarios))
}


for scenario_idx, scenario in enumerate(scenario_params):

    L_var = scenario['L_var']
    max_inter = scenario['max_inter']

    for seed in range(1, n_seeds + 1):
        np.random.seed(seed)  # set seed
        # RNG for interaction matrix (changes across seeds)
        rng = np.random.default_rng(seed)

        # impose lysis variability
        sigmas = sigma + L_var * generate_valid_random(N)
        lambdas = lambda_val + L_var * generate_valid_random(N)
        # residue parameters
        li = np.clip(np.random.normal(0,0.2,N),0,1)
        de = np.clip(np.random.normal(0, 0.1, N), 0, None)
                
        # generate interaction matrix
        current_max_inter = max_Gs[-1] * max_inter if max_inter > 0 else 0
        params = param_generator_lv(N, connectedness, neg_frac, 
                                    max_neg=current_max_inter, 
                                    max_pos=current_max_inter * 0.5, rng=rng)

        # simulate
        t, y, gr, egr, lr, resid, b_with, gr_with, lr_with = simulate_timecourse_with_rates(x0, tend, params, max_Gs, sigmas, lambdas, li, de)

        # store into outputs
        outputs['population_data'][:, :, seed-1, scenario_idx] = y
        outputs['growth_rates'][:, :, seed-1, scenario_idx] = gr
        outputs['effective_growth_rates'][:, :, seed-1, scenario_idx] = egr
        outputs['lysis_rates'][:, :, seed-1, scenario_idx] = lr
        outputs['relative_abundance'][:, :, seed-1, scenario_idx] = y / y.sum(axis=1, keepdims=True)
        outputs['residue'][:, :, seed-1, scenario_idx] = resid
        outputs['biomass_with_resid'][:, :, seed-1, scenario_idx] = b_with
        outputs['net_rate_with_resid'][:, :, seed-1, scenario_idx] = gr_with
        outputs['lysis_rate_with_resid'][:, :, seed-1, scenario_idx] = lr_with
        

# Save full dataset
np.savez(f'simulation_results_{n_seeds}seeds_with_debris.npz',
         time_points=t,
         population_data=outputs['population_data'],
         growth_rates=outputs['growth_rates'],
         effective_growth_rates=outputs['effective_growth_rates'],
         lysis_rates=outputs['lysis_rates'],
         relative_abundance=outputs['relative_abundance'],
         residue=outputs['residue'],
         biomass_with_resid=outputs['biomass_with_resid'],
         net_rate_with_resid=outputs['net_rate_with_resid'],
         lysis_rate_with_resid=outputs['lysis_rate_with_resid'],
         scenario_params=scenario_params)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import csv

# -------------------- config --------------------
n_seeds = 100
npz_path = f"simulation_results_{n_seeds}seeds_with_debris.npz"
fit_hours   = np.array([4, 5, 6, 8, 10])
eval_hours  = np.array([5, 6])
colors = {5: "#64A9D3", 6: "#216FB0"}
scenario_params = [{'L_var': lv, 'max_inter': mi}
                   for lv in [0, 0.1, 0.2]
                   for mi in [1, 5, 10]]
n_scenarios = len(scenario_params)

# ---- font sizes (balanced) ----
FS_SUPTITLE = 16
FS_TITLE    = 12
FS_LABEL    = 12
FS_LEGEND   = 12
FS_TICKS    = 12
FS_TEXTBOX  = 12
SCAT_SIZE   = 14

# -------------------- load --------------------
npz = np.load(npz_path, allow_pickle=True)
t  = npz["time_points"]          # (T,)
pop = npz["population_data"]     # 4D with a seed axis somewhere
gr  = npz["growth_rates"]        # same shape as pop
lr_true  = npz["lysis_rates"]
lr_with  = npz["lysis_rate_with_resid"]
T = t.size

def to_seed_time_species_scenario(arr, n_seeds, T, n_scenarios):
    """Reorder to (seed, time, species, scenario)."""
    if arr.ndim != 4:
        raise ValueError(f"Need 4D array, got {arr.shape}")
    shape = arr.shape
    idx_seed = [i for i,s in enumerate(shape) if s == n_seeds][0]
    idx_time = [i for i,s in enumerate(shape) if s == T and i != idx_seed][0]
    idx_scen = [i for i,s in enumerate(shape) if s == n_scenarios and i not in (idx_seed, idx_time)][0]
    idx_spec = [ax for ax in range(4) if ax not in (idx_seed, idx_time, idx_scen)][0]
    return np.transpose(arr, (idx_seed, idx_time, idx_spec, idx_scen))

pop4 = to_seed_time_species_scenario(pop, n_seeds, T, n_scenarios)
gr4  = to_seed_time_species_scenario(gr,  n_seeds, T, n_scenarios)
lr_true4 = to_seed_time_species_scenario(lr_true, n_seeds, T, n_scenarios)
lr_with4 = to_seed_time_species_scenario(lr_with, n_seeds, T, n_scenarios)
n_seed, _, n_species, _ = pop4.shape

# indices for fit & eval
fit_idx  = np.array([np.argmin(np.abs(t - h)) for h in fit_hours])
eval_idx = np.array([np.argmin(np.abs(t - h)) for h in eval_hours])

# -------------------- plotting --------------------
fig, axes = plt.subplots(3, 3, figsize=(10, 10))   # Use a compact layout


with open("R2_per_seed_by_scenario.csv", "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["scenario_index","L_var","max_inter","seed","R2"])

    for scen in range(n_scenarios):
        ax = axes[scen // 3, scen % 3]
        r2_list = []
        plot_true = {h: [] for h in eval_hours}
        plot_fit  = {h: [] for h in eval_hours}

        for sd in range(n_seed):
            y   = pop4[sd, :, :, scen]     # (T, S)
            tru = lr_true4[sd, :, :, scen]
            fit = lr_with4[sd, :, :, scen]

            seed_true_all, seed_fit_all = [], []

            for sp in range(n_species):
                for h, idx in zip(eval_hours, eval_idx):
                    tval = tru[idx, sp]
                    fval = fit[idx, sp]

                    if np.isfinite(fval) and np.isfinite(tval):
                        plot_true[h].append(tval)
                        plot_fit[h].append(fval)
                        seed_true_all.append(tval)
                        seed_fit_all.append(fval)

            if len(seed_true_all) > 1:
                r2 = r2_score(seed_true_all, seed_fit_all)
                r2_list.append(r2)
                writer.writerow([scen+1, scenario_params[scen]["L_var"],
                                 scenario_params[scen]["max_inter"], sd, r2])

        # scatter all seeds (color by hour)
        for h in eval_hours:
            ax.scatter(plot_true[h], plot_fit[h], alpha=0.35, s=SCAT_SIZE,
                       label=f"{h} h", color=colors[h])

        # identity line & limits
        if plot_true[eval_hours[0]]:
            all_true = np.concatenate([np.array(plot_true[h]) for h in eval_hours])
            all_fit  = np.concatenate([np.array(plot_fit[h])  for h in eval_hours])
            lo = np.floor(min(all_true.min(), all_fit.min()) - 0.1)
            hi = np.ceil(max(all_true.max(), all_fit.max()) + 0.1)
            ax.plot([lo, hi], [lo, hi], "k--", alpha=0.7, linewidth=1)

        # mean ± SD R² text
        if r2_list:
            ax.text(0.04, 0.96,
                    f"R² = {np.mean(r2_list):.3f} ",
                    transform=ax.transAxes, ha="left", va="top",
                    bbox=dict(facecolor="white", alpha=0.85),
                    fontsize=FS_TEXTBOX)
        else:
            ax.text(0.04, 0.96, "R² across seeds: NA",
                    transform=ax.transAxes, ha="left", va="top",
                    bbox=dict(facecolor="white", alpha=0.85),
                    fontsize=FS_TEXTBOX)

        ax.set_title(
            f"Scenario {scen+1}\nL_var={scenario_params[scen]['L_var']}, "
            f"max_inter={scenario_params[scen]['max_inter']}",
            fontsize=FS_TITLE
        )
        ax.set_xlabel("True lysis rate")
        ax.set_ylabel("OD-derived lysis rate with residue")
        ax.tick_params(axis='both', which='major', labelsize=FS_TICKS)
        ax.grid(alpha=0.2)

        if scen == 0:
            ax.legend(loc="lower right", fontsize=FS_LEGEND)

plt.tight_layout()
plt.savefig(f'True_vs_debris_lysis_rate_{n_seeds}seeds_eval56.pdf', dpi=300)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import csv

# -------------------- config --------------------
n_seeds = 100
npz_path = f"simulation_results_{n_seeds}seeds_with_debris.npz"

fit_hours   = np.array([4, 5, 6, 8, 10])
eval_hours  = np.array([5, 6])

colors = {5: "#64A9D3", 6: "#216FB0"}

scenario_params = [{'L_var': lv, 'max_inter': mi}
                   for lv in [0, 0.1, 0.2]
                   for mi in [1, 5, 10]]

n_scenarios = len(scenario_params)

# number of seeds used for plotting
n_plot_seeds = 10
rng = np.random.default_rng(42)

# ---- plot parameters ----
FS_TEXTBOX = 12
FS_LEGEND = 12
FS_TICKS = 12
SCAT_SIZE = 14


# -------------------- load data --------------------
npz = np.load(npz_path, allow_pickle=True)

t  = npz["time_points"]
pop = npz["population_data"]
gr  = npz["growth_rates"]
lr_true  = npz["lysis_rates"]
lr_with  = npz["lysis_rate_with_resid"]

T = t.size


# -------------------- reshape helper --------------------
def to_seed_time_species_scenario(arr, n_seeds, T, n_scenarios):

    shape = arr.shape

    idx_seed = [i for i,s in enumerate(shape) if s == n_seeds][0]
    idx_time = [i for i,s in enumerate(shape) if s == T and i != idx_seed][0]
    idx_scen = [i for i,s in enumerate(shape) if s == n_scenarios and i not in (idx_seed, idx_time)][0]
    idx_spec = [ax for ax in range(4) if ax not in (idx_seed, idx_time, idx_scen)][0]

    return np.transpose(arr, (idx_seed, idx_time, idx_spec, idx_scen))


pop4 = to_seed_time_species_scenario(pop, n_seeds, T, n_scenarios)
lr_true4 = to_seed_time_species_scenario(lr_true, n_seeds, T, n_scenarios)
lr_with4 = to_seed_time_species_scenario(lr_with, n_seeds, T, n_scenarios)

n_seed, _, n_species, _ = pop4.shape


# randomly select seeds for plotting
seed_indices = np.sort(rng.choice(n_seed, size=n_plot_seeds, replace=False))

# indices for evaluation hours
fit_idx  = np.array([np.argmin(np.abs(t - h)) for h in fit_hours])
eval_idx = np.array([np.argmin(np.abs(t - h)) for h in eval_hours])


# -------------------- plotting --------------------
fig, axes = plt.subplots(3, 3, figsize=(7, 7))


csv_name = f"Pearson_per_seed_subset_{n_plot_seeds}of{n_seed}.csv"

with open(csv_name, "w", newline="") as fh:

    writer = csv.writer(fh)

    writer.writerow([
        "scenario_index",
        "L_var",
        "max_inter",
        "seed",
        "Pearson_r",
        "p_value"
    ])

    for scen in range(n_scenarios):

        ax = axes[scen // 3, scen % 3]

        r_list = []
        p_list = []

        plot_true = {h: [] for h in eval_hours}
        plot_fit  = {h: [] for h in eval_hours}

        for sd in seed_indices:

            y   = pop4[sd, :, :, scen]
            tru = lr_true4[sd, :, :, scen]
            fit = lr_with4[sd, :, :, scen]

            seed_true_all = []
            seed_fit_all  = []

            for sp in range(n_species):

                for h, idx in zip(eval_hours, eval_idx):

                    tval = tru[idx, sp]
                    fval = fit[idx, sp]

                    if np.isfinite(fval) and np.isfinite(tval):

                        plot_true[h].append(tval)
                        plot_fit[h].append(fval)

                        seed_true_all.append(tval)
                        seed_fit_all.append(fval)

            if len(seed_true_all) > 1:

                r, p = pearsonr(seed_true_all, seed_fit_all)

                r_list.append(r)
                p_list.append(p)

                writer.writerow([
                    scen + 1,
                    scenario_params[scen]["L_var"],
                    scenario_params[scen]["max_inter"],
                    sd,
                    r,
                    p
                ])


        # ---------------- scatter plot ----------------
        for h in eval_hours:

            ax.scatter(
                plot_true[h],
                plot_fit[h],
                alpha=0.35,
                s=SCAT_SIZE,
                label=f"{h} h",
                color=colors[h]
            )


        # ---------------- identity line ----------------
        if plot_true[eval_hours[0]]:

            ax.plot([0,4],[0,4],"k--",alpha=0.7,linewidth=1)

            ax.set_xlim(0,4)
            ax.set_ylim(0,4)

            ax.set_xticks([0,1,2,3,4])
            ax.set_yticks([0,1,2,3,4])

            ax.set_aspect('equal')


        # ---------------- show correlation ----------------
        if r_list:

            ax.text(
                0.04,
                0.96,
                f"R = {np.mean(r_list):.3f}",
                transform=ax.transAxes,
                ha="left",
                va="top",
                bbox=dict(facecolor="white", alpha=0.85),
                fontsize=FS_TEXTBOX
            )


        ax.tick_params(axis='both', labelsize=FS_TICKS)
        ax.grid(alpha=0.2)

        if scen == 0:
            ax.legend(loc="lower right", fontsize=FS_LEGEND)


plt.tight_layout()

plt.savefig(
    f"Pearson_True_vs_debris_lysis_rate_subset_{n_plot_seeds}of{n_seed}_eval56.pdf",
    dpi=300
)

plt.show()

In [ ]:
import numpy as np
from scipy.stats import linregress
from sklearn.metrics import r2_score

# Load simulation data
data = np.load(f'simulation_results_{n_seeds}seeds_with_debris.npz', allow_pickle=True)

effective_growth_rates = data["effective_growth_rates"]
lysis_rates = data["lysis_rates"]
lysis_rate_with_resid = data["lysis_rate_with_resid"]

scenario_params = data["scenario_params"]
time_points = data["time_points"]

n_time, n_species, n_seeds, n_scenarios = effective_growth_rates.shape

# Target time
time_target = 6
time_index = np.argmin(np.abs(time_points - time_target))

results = []

for scenario_idx in range(n_scenarios):

    L_var = scenario_params[scenario_idx]['L_var']
    max_inter = scenario_params[scenario_idx]['max_inter']
    
    for seed_idx in range(n_seeds):

        x = effective_growth_rates[time_index, :, seed_idx, scenario_idx]
        y_true = lysis_rates[time_index, :, seed_idx, scenario_idx]
        y_with = lysis_rate_with_resid[time_index, :, seed_idx, scenario_idx]

        # ---------- G vs L_true ----------
        res = linregress(x, y_true)

        slope = res.slope
        intercept = res.intercept
        p_value = res.pvalue

        y_pred = slope * x + intercept
        r2 = r2_score(y_true, y_pred)

        results.append([
            "L_true",
            L_var,
            max_inter,
            seed_idx,
            time_points[time_index],
            r2,
            slope,
            intercept,
            p_value
        ])

        # ---------- G vs L_with ----------
        res = linregress(x, y_with)

        slope = res.slope
        intercept = res.intercept
        p_value = res.pvalue

        y_pred = slope * x + intercept
        r2 = r2_score(y_with, y_pred)

        results.append([
            "L_with",
            L_var,
            max_inter,
            seed_idx,
            time_points[time_index],
            r2,
            slope,
            intercept,
            p_value
        ])

# Save results
header = "type\tL_var\tmax_inter\tseed\ttime_point\tR2\tslope\tintercept\tp_value"
output_path = f'linear_fitting_results_G_vs_L_true_and_with_residue_{n_seeds}seeds.txt'

with open(output_path, "w") as f:
    f.write(header + "\n")
    for row in results:
        f.write("\t".join(
            f"{val:.6e}" if isinstance(val,float) else str(val)
            for val in row
        ) + "\n")

print(f"Saved results to {output_path}")


In [ ]:
# --- must define n_seeds before running, e.g. n_seeds = 100

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load data
file_path = f'linear_fitting_results_G_vs_L_true_and_with_residue_{n_seeds}seeds.txt'
df = pd.read_csv(file_path, sep="\t")

# Nature-inspired color palette
nature_palette = {
    "L_true": "#E69F00",   # orange
    "L_with": "#0072B2"    # blue
}

# -------- style --------
sns.set_style("white")                
sns.set_context("talk", font_scale=1.2)  

fig, axes = plt.subplots(1,2, figsize=(10,5))


# ---------------- slope density ----------------
sns.kdeplot(
    data=df,
    x="slope",
    hue="type",
    fill=True,
    common_norm=False,
    palette=nature_palette,
    alpha=0.4,
    ax=axes[0]
)

axes[0].set_title("Slope distribution", fontsize=18)
axes[0].set_xlabel("Slope", fontsize=16)
axes[0].set_ylabel("Density", fontsize=16)


# ---------------- intercept density ----------------
sns.kdeplot(
    data=df,
    x="intercept",
    hue="type",
    fill=True,
    common_norm=False,
    palette=nature_palette,
    alpha=0.4,
    ax=axes[1]
)

axes[1].set_title("Intercept distribution", fontsize=18)
axes[1].set_xlabel("Intercept", fontsize=16)
axes[1].set_ylabel("Density", fontsize=16)


# -------- tick size --------
for ax in axes:
    ax.tick_params(axis='both', labelsize=14)
    
# -------- unify y-axis --------
ymax = max(axes[0].get_ylim()[1], axes[1].get_ylim()[1])
axes[0].set_ylim(0, ymax)
axes[1].set_ylim(0, ymax)

plt.tight_layout()

plt.savefig(f'slope_intercept_density_{n_seeds}seeds.pdf', dpi=300)

plt.show()